# LAB 3: Prompt Engineering Patterns & Conversation Memory

### Objective
To demonstrate prompt engineering patterns and the use of **conversation memory** in LangChain using a Hugging Face LLM.

**Platform:** Google Colab
**Tools:** LangChain, Hugging Face Transformers


## STEP 1: Install Required Libraries
Run this cell once to install dependencies.

In [ ]:
!pip install -q langchain langchain-community langchain-core transformers accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## STEP 2: Load Hugging Face LLM (FLAN-T5)
We use a lightweight open-source model suitable for Colab.

In [ ]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

hf_pipeline = pipeline(
    task='text2text-generation',
    model='google/flan-t5-base',
    max_length=256
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu
/tmp/ipython-input-4277564332.py:10: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=hf_pipeline)


## STEP 3: Import PromptTemplate

In [ ]:
from langchain_core.prompts import PromptTemplate

## STEP 4: Conversation WITHOUT Memory (Problem Demonstration)
Each question is treated independently.

In [ ]:
simple_prompt = PromptTemplate(
    input_variables=['question'],
    template='Answer the question:\n{question}'
)

print(llm.invoke(simple_prompt.format(question='Who is Donald Trump?')))
print(llm.invoke(simple_prompt.format(question='What is his wife name?')))

president of the united states
elizabeth harris


## STEP 5: Conversation WITH Memory (Solution)
LangChain memory allows the model to remember previous interactions.

In [ ]:
# Correct imports for latest LangChain
from langchain_core.memory import ConversationBufferMemory
from langchain.chains import LLMChain


ModuleNotFoundError: No module named 'langchain_core.memory'

In [ ]:
memory = ConversationBufferMemory(
    memory_key='chat_history',
    return_messages=True
)

In [ ]:
memory_prompt = PromptTemplate(
    input_variables=['chat_history', 'question'],
    template='''
You are a helpful AI assistant.

Conversation so far:
{chat_history}

User question:
{question}

Answer:
'''
)

In [ ]:
memory_chain = LLMChain(
    llm=llm,
    prompt=memory_prompt,
    memory=memory
)

In [ ]:
response1 = memory_chain.invoke({'question': 'Who is Donald Trump?'})
print(response1['text'])

response2 = memory_chain.invoke({'question': 'What is his wife name?'})
print(response2['text'])

## STEP 6: Inspect Stored Memory
This shows that previous conversation is retained.

In [ ]:
print(memory.buffer)

## STEP 7: Comparison Summary

| Feature | Without Memory | With Memory |
|--------|---------------|------------|
| Context Retention | ❌ No | ✅ Yes |
| Follow-up Questions | ❌ Incorrect | ✅ Correct |
| Multi-turn Chat | ❌ No | ✅ Yes |


## Viva / Exam Note
**ConversationBufferMemory** stores previous interactions and appends them to the prompt, enabling context-aware responses in multi-turn conversations.